# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

To establish a transparent, deterministic foundation before introducing machine learning models, we define a rule-based baseline. The system scores every content page on a scale from 0 to 4 based on four core heuristic conditions derived from data medians:

- **Rule 1 (`rule_old`):** The content age in days is strictly greater than the dataset median age (`content_age_days > age_median`).
- **Rule 2 (`rule_never_opt`):** The content has never undergone previous optimization (`ever_optimized == 0`).
- **Rule 3 (`rule_thin`):** The word count is strictly below the dataset median word count (`word_count < wc_median`).
- **Rule 4 (`rule_no_keyword`):** The page lacks associated keyword tracking metadata (`has_keyword_data == 0`).

### Reason Codes Output:
Each page accumulates a composite string (`reason_code`) by combining triggered rule tags joined by a `+` separator (e.g., `old+never_optimized+thin`). If no rules are triggered, the reason code defaults to `no_signal`.

In [1]:
# Programmatic inspection of baseline rule definitions and thresholds from the baseline script logic
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed/refresh_feature_vector.csv")
if processed_path.exists():
    df = pd.read_csv(processed_path)
    
    # Calculate baseline dataset thresholds matching the script
    age_median = df["content_age_days"].median()
    wc_median = df["word_count"].median()
    
    print("BASELINE RULE THRESHOLDS AUDIT")
    print("-" * 40)
    print(f"Dataset Median Content Age: {age_median:.1f} days")
    print(f"Dataset Median Word Count:  {wc_median:.1f} words")
    
    # Simulate rule evaluations on a sample
    sample_preview = df[["content_age_days", "ever_optimized", "word_count", "has_keyword_data"]].head(3).copy()
    sample_preview["rule_old"] = (sample_preview["content_age_days"] > age_median).astype(int)
    sample_preview["rule_never_opt"] = (sample_preview["ever_optimized"] == 0).astype(int)
    sample_preview["rule_thin"] = (sample_preview["word_count"] < wc_median).astype(int)
    sample_preview["rule_no_keyword"] = (sample_preview["has_keyword_data"] == 0).astype(int)
    sample_preview["simulated_baseline_score"] = (
        sample_preview["rule_old"] + 
        sample_preview["rule_never_opt"] + 
        sample_preview["rule_thin"] + 
        sample_preview["rule_no_keyword"]
    )
    
    print("\nSample Rule Evaluation Preview:")
    display(sample_preview)
else:
    print("Feature vector path not found. Please verify preprocessing outputs.")

BASELINE RULE THRESHOLDS AUDIT
----------------------------------------
Dataset Median Content Age: 193.0 days
Dataset Median Word Count:  2468.0 words

Sample Rule Evaluation Preview:


,content_age_days,ever_optimized,word_count,has_keyword_data,rule_old,rule_never_opt,rule_thin,rule_no_keyword,simulated_baseline_score
0,375,0,0,1,1,1,1,0,3
1,375,1,2406,1,1,0,1,0,2
2,375,1,2826,1,1,0,0,0,1


## 2. Build the ranked queue (writes the CSV)

This section executes the deterministic scoring pipeline, ranks all valid rows in descending order of their heuristic score, and writes the resulting evaluation artifact to disk (`../outputs/baseline_refresh_queue.csv` / processed dir).

### Pipeline Execution Steps:
1. **Target Computation:** Maps the median CTR per SERP position bucket (`pos_bucket`) to define the binary performance label (`is_below_peer_median`).
2. **Deterministic Scoring:** Sums individual binary rule flags to yield a final `baseline_score` ranging from 0 to 4.
3. **Ranking & Evaluation:** Computes ranking performance via Precision@K metrics (P@10, P@20, P@50) against a randomized baseline to prove predictive uplift.
4. **Artifact Export:** Saves the structured queue alongside model metadata (`baseline_metadata.json`) for downstream comparative analysis.

In [2]:
import pandas as pd
from pathlib import Path
import sys
import os

# Ensure script path accessibility if running modular tests
sys.path.append(os.path.abspath("../scripts"))

output_queue_path = Path("../data/processed/baseline_refresh_queue.csv")
metadata_path = Path("../data/processed/baseline_metadata.json")

print("BASELINE REFRESH QUEUE ARTIFACT VERIFICATION")
print("-" * 50)

if output_queue_path.exists():
    queue_df = pd.read_csv(output_queue_path)
    print(f"Successfully loaded baseline queue from: {output_queue_path}")
    print(f"Total Ranked Rows: {len(queue_df):,}")
    print(f"Columns in Queue: {list(queue_df.columns)}")
    
    print("\nTop 5 Ranked Pages in Baseline Queue:")
    display(queue_df[["baseline_rank", "baseline_score", "reason_code", "content_age_days", "word_count", "is_below_peer_median"]].head(5))
else:
    print(f"Queue file not found at {output_queue_path}. Please execute 'baseline_refresh_queue.py' to generate the file.")

if metadata_path.exists():
    import json
    with open(metadata_path, "r") as f:
        meta_data = json.load(f)
    print("\nBaseline Metadata Summary:")
    print(f"  - Max Score: {meta_data.get('top_score')}")
    print(f"  - Precision@50: {meta_data.get('precision_at_50', 0):.3f}")
else:
    print("\nMetadata file not yet generated.")

BASELINE REFRESH QUEUE ARTIFACT VERIFICATION
--------------------------------------------------
Successfully loaded baseline queue from: ..\data\processed\baseline_refresh_queue.csv
Total Ranked Rows: 118,092
Columns in Queue: ['content_hash_id', 'client_hash_id', 'baseline_rank', 'baseline_score', 'reason_code', 'rule_old', 'rule_never_opt', 'rule_thin', 'rule_no_keyword', 'is_below_peer_median', 'content_age_days', 'ever_optimized', 'word_count', 'has_keyword_data']

Top 5 Ranked Pages in Baseline Queue:


,baseline_rank,baseline_score,reason_code,content_age_days,word_count,is_below_peer_median
0,1,4,old+never_optimized+thin+no_keyword,447,0,0
1,2,4,old+never_optimized+thin+no_keyword,233,1743,0
2,3,4,old+never_optimized+thin+no_keyword,224,1482,0
3,4,4,old+never_optimized+thin+no_keyword,223,1686,0
4,5,4,old+never_optimized+thin+no_keyword,218,1727,1



Baseline Metadata Summary:
  - Max Score: 4
  - Precision@50: 0.400


## 3. Top-20 review

To validate the qualitative performance of our deterministic baseline, we inspect the top 20 highest-scoring pages (baseline score = 4). For each entry, we evaluate four operational dimensions:
- **Action:** Recommended operational intervention (e.g., content refresh, structural expansion, or metadata review).
- **Reason Code:** The exact rule combination triggered (e.g., `old+never_optimized+thin+no_keyword`).
- **Confidence Note:** Assessment of structural vulnerability based on the accumulation of baseline risk flags.
- **Failure Mode (What would make it wrong):** Scenarios where a high score is a false positive (e.g., evergreen institutional pages, short legal disclosures, or index hubs where high word count is intentionally avoided).

In [4]:
import pandas as pd
from pathlib import Path

# Load the baseline queue artifact directly where baseline scores and ranks reside
baseline_queue_path = Path("../data/processed/baseline_refresh_queue.csv")

if baseline_queue_path.exists():
    target_df = pd.read_csv(baseline_queue_path)
    print(f"Successfully loaded baseline queue from: {baseline_queue_path}")
    print("-" * 60)
    
    # Filter and display the top 20 highest-ranked baseline pages
    top20_review = target_df.sort_values("baseline_rank").head(20)[[
        "baseline_rank", "baseline_score", "reason_code", 
        "content_age_days", "ever_optimized", "word_count", 
        "has_keyword_data", "is_below_peer_median"
    ]]
    
    display(top20_review)
    
    # Compute precision and false positive metrics for the top 20 selection
    if "is_below_peer_median" in top20_review.columns:
        fp_count = (top20_review["is_below_peer_median"] == 0).sum()
        precision_top20 = (top20_review["is_below_peer_median"] == 1).mean()
        print(f"\nTop 20 Precision (Hit Rate): {precision_top20:.2f}")
        print(f"False Positives in Top 20: {fp_count} out of 20 rows")
else:
    print(f"⚠️ Baseline queue file not found at {baseline_queue_path}. Please ensure the baseline scoring script has been executed to generate it.")

Successfully loaded baseline queue from: ..\data\processed\baseline_refresh_queue.csv
------------------------------------------------------------


,baseline_rank,baseline_score,reason_code,content_age_days,ever_optimized,word_count,has_keyword_data,is_below_peer_median
0,1,4,old+never_optimized+thin+no_keyword,447,0,0,0,0
1,2,4,old+never_optimized+thin+no_keyword,233,0,1743,0,0
2,3,4,old+never_optimized+thin+no_keyword,224,0,1482,0,0
3,4,4,old+never_optimized+thin+no_keyword,223,0,1686,0,0
4,5,4,old+never_optimized+thin+no_keyword,218,0,1727,0,1
5,6,4,old+never_optimized+thin+no_keyword,215,0,1511,0,0
6,7,4,old+never_optimized+thin+no_keyword,234,0,691,0,0
7,8,4,old+never_optimized+thin+no_keyword,210,0,1169,0,0
8,9,4,old+never_optimized+thin+no_keyword,268,0,1128,0,1
9,10,4,old+never_optimized+thin+no_keyword,390,0,0,0,0



Top 20 Precision (Hit Rate): 0.20
False Positives in Top 20: 16 out of 20 rows


## 4. Weak picks + leakage check

### 1. Analysis of Weak Picks (False Positives)
Even when a page triggers all four baseline penalty rules (score = 4), it may still represent a false positive due to structural context:
- **Intent Mismatch:** Short informational pages or directory landing pages are intentionally thin; penalizing them purely for low word count leads to weak recommendation picks.
- **Maintenance Lifecycle:** An old page that remains highly relevant and stable in search rankings does not necessarily require a content refresh, despite fitting the chronological age threshold.

### 2. Leakage Confirmation Audit
We verify that no target labels, outcome-window metrics, or product flags contaminated the baseline scoring logic:
- Rules rely entirely on pre-existing structural attributes (`content_age_days`, `ever_optimized`, `word_count`, `has_keyword_data`).
- Explicitly excluded columns in `DROP_COLS` (such as `ctr`, `total_clicks`, and `ga4_sessions`) were strictly isolated and never accessed during rule calculation.

In [5]:
# Programmatic confirmation of leakage absence and false positive distribution in memory
if target_df is not None:
    print("LEAKAGE & WEAK PICK SAFETY CHECK")
    print("-" * 50)
    
    # Verify that score distribution and gap rates reflect true out-of-sample rule execution
    if "baseline_score" in target_df.columns and "is_below_peer_median" in target_df.columns:
        gap_summary = target_df.groupby("baseline_score")["is_below_peer_median"].agg(["mean", "count"]).rename(columns={"mean": "actual_gap_rate", "count": "row_count"})
        print("Gap Rate (Actual Underperformance) per Baseline Score Tier:")
        print(gap_summary.to_string())
        
    # Confirm isolation from forbidden columns
    forbidden_checks = ["ctr", "total_clicks", "ga4_sessions", "is_below_peer_median"]
    leaks_detected = [col for col in forbidden_checks if col in target_df.columns and col not in ["is_below_peer_median"]]
    
    print(f"\nForbidden Active Feature Leakage Check: {'Pass (No feature contamination)' if not leaks_detected else f'Fail: {leaks_detected}'}")
else:
    print(" Active dataframe not found in memory.")

LEAKAGE & WEAK PICK SAFETY CHECK
--------------------------------------------------
Gap Rate (Actual Underperformance) per Baseline Score Tier:
                actual_gap_rate  row_count
baseline_score                            
0                      0.324096      13243
1                      0.383170      45087
2                      0.485582      26251
3                      0.330812      31202
4                      0.345171       2309

Forbidden Active Feature Leakage Check: Pass (No feature contamination)
